In [5]:
import os
import json
import numpy as np
import pandas as pd

## Specity the Inputs

In [38]:
# specify your path of prescient file here
this_file_path = os.getcwd()
prescient_file_path = os.path.join("your_path")

# the json file is the one used to store the information of the generators
json_path = os.path.join("..", "Data", "gen_dict.json")

# for information of the bus and generator names, please visit 
# https://github.com/GridMod/RTS-GMLC/tree/master/RTS_Data/SourceData
bus_name = "Abel"
gen_name = "101_CT_1"

## Check the Inputs

In [40]:
# Read the generator information from my files
with open(json_path, "rb") as f:
    gen_dict = json.load(f)
gen_dict["fossil"][gen_name].keys()

dict_keys(['name', 'bus_name', 'gen_type', 'max_p', 'min_p', 'ramp', 'fuel_p', 'min_down_time', 'min_up_time', 'start_up_time_hot', 'start_up_time_warm', 'start_up_time_cold', 'start_heat_hot', 'start_heat_warm', 'start_heat_cold', 'cost_curve'])

In [30]:
# check the if the generator is in the bus

def check_generator_and_bus(bus_name, gen_name):
    # check which type generator that the generator belongs to (renewable or fossil)
    if gen_name in gen_dict["fossil"].keys():
        gen_type = "fossil"
    elif gen_name in gen_dict["renew"].keys():
        gen_type = "renew"
    else:
        raise ValueError("The generator name is not vaild")
    
    # check the if the generator is in the bus provided.
    bus_name_ = gen_dict[gen_type][gen_name]["bus_name"]
    if bus_name == bus_name_:
        print(f"The generator {gen_name} is a {gen_type} generator. \nThe generator {gen_name} is in the bus {bus_name}.")
    
    else:
        raise ValueError(f"The generator {gen_name} should be located at bus {bus_name_}, \n but bus {bus_name} is provided.")
    
    return gen_type

In [36]:
# run this cell to make sure your are reading the correct bus and generator information.
gen_type = check_generator_and_bus(bus_name, gen_name)

The generator 101_CT_1 is a fossil generator. 
The generator 101_CT_1 is in the bus Abel.


## Read LMP and Dispatch Results

In [45]:
# read Prescient to pandas dataframe
def _prescient_output_to_df(file_name):
    '''Helper for loading data from Prescient output csv.
        Combines Datetimes into single column.
    '''
    df = pd.read_csv(file_name)
    df['Datetime'] = \
        pd.to_datetime(df['Date']) + \
        pd.to_timedelta(df['Hour'], 'hour') + \
        pd.to_timedelta(df['Minute'], 'minute')
    df.drop(columns=['Date','Hour','Minute'], inplace=True)
    # put 'Datetime' in front
    cols = df.columns.tolist()
    cols = cols[-1:]+cols[:-1]
    
    return df[cols]

In [47]:
# read the lmp at the bus and put it in a csv
def make_lmp_csv(lmp_path, bus_details_path, bus_name):
    """
    This function reads all the LMP at the bus and put it into a dataframe.
    You can loop over all the buses and get all results in one csv.
    
    Args:
        lmp_path: str (path), the path for the csv you want to save the lmp.
        bus_details_path: str (path), the path of bus_detail.csv in the prescient results.
        bus_name: str, the name of the bus.
    """
    bdf = _prescient_output_to_df(bus_details_path)
    bdf = bdf[bdf["Bus"] == bus_name][["Datetime","LMP","LMP DA"]]
    bdf.set_index("Datetime", inplace=True)

    if lmp_path == None:
        print("Empty path, make a new df")
        bdf = bdf.rename(columns={'LMP': f'{bus_name}_LMP', "LMP DA": f'{bus_name}_LMP_DA'})
        lmp_df = bdf
        lmp_df.to_csv("Bus_LMP.csv")
    else:
        print(f"Extracting LMP for {bus_name}")
        bdf = bdf.rename(columns={"LMP": f"{bus_name}_LMP", "LMP DA": f"{bus_name}_LMP_DA"})
        lmp_df = pd.read_csv(lmp_path).set_index("Datetime")
        # check if the bus has already been read
        if (f"{bus_name}_LMP" in lmp_df.columns) or (f"{bus_name}_LMP_DA" in lmp_df.columns):
            print(f"{bus_name} LMP already exists.")
        else:
            bdf_aligned = bdf.reindex(lmp_df.index)
            lmp_df_merge = pd.concat([lmp_df, bdf_aligned], axis=1)
            lmp_df_merge.to_csv(lmp_path)
    
    return

In [63]:
# read the dispatch at the bus and put it in a csv
def make_dispatch_csv(dispatch_path, gen_details_path, gen_name, gen_type=gen_type, other_info=None):
    """
    This function reads all the dispatch result of the generator and put it into a dataframe.
    You can loop over all the buses and get all results in one csv.
    
    Args:
        dispatch_path: str (path), the path for the csv you want to save the dispatch results.
        gen_details_path: str (path), the path of thermal_detail.csv or renewables_detail.csv in the prescient results.
        gen_name: str, the name of the generator.
        gen_type: str, fossil or renew, the result from Prescient varies from generator types.
        other_info: list, ["Curtailment"], the information you want to read, such as Curtailment, Unit Cost
    """, 
    gdf = _prescient_output_to_df(gen_details_path)
    if gen_type == "fossil":
        info_list = ["Datetime","Dispatch","Dispatch DA"]
    if gen_type == "renew":
        info_list = ["Datetime","Output","Output DA"]
    # add other information you want to besides the default DA/RT dispatch
    if other_info is not None: 
        for info in other_info:
            info_list.append(info)
    gdf = gdf[gdf["Generator"] == gen_name][info_list]
    gdf.set_index("Datetime", inplace=True)
    
    # rename the columns by adding the generator name.
    new_col_name = {}
    for i in info_list:
        new_col_name[i] = f"{gen_name}_{i}"
    
    if dispatch_path == None:
        print("Empty path, make a new df")
        gdf = gdf.rename(columns=new_col_name)
        dispatch_df = gdf
        dispatch_df.to_csv("Generator_Dispatch.csv")
    else:
        print(f"Extracting LMP for {gen_name}")
        gdf = gdf.rename(columns=new_col_name)
        dispatch_df = pd.read_csv(dispatch_path).set_index("Datetime")
        # check if the bus has already been read
        for j in info_list:
            if f'{gen_name}_{j}' in dispatch_df.columns:
                print(f"{gen_name}_{j} already exists.")
            else:
                # merge the previous and current one
                gdf_aligned = gdf.reindex(dispatch_df.index)
                dispatch_df_merge = pd.concat([dispatch_df, gdf_aligned], axis=1)
                dispatch_df_merge.to_csv(dispatch_path)
    
    return

In [49]:
# Here, iterate over all the fossil generators to get dispatch and LMP

# LMP
for idx, key in enumerate(list(gen_dict["fossil"].keys())):
    if idx == 0:
        make_lmp_csv(lmp_path=None, bus_details_path="../Data/bus_detail.csv", bus_name=gen_dict["fossil"][key]["bus_name"])
    else:
        make_lmp_csv(lmp_path="Bus_LMP.csv", bus_details_path="../Data/bus_detail.csv", bus_name=gen_dict["fossil"][key]["bus_name"])

Empty path, make a new df
Extracting LMP for Abel
Abel LMP already exists.
Extracting LMP for Abel
Abel LMP already exists.
Extracting LMP for Abel
Abel LMP already exists.
Extracting LMP for Adams
Extracting LMP for Adams
Adams LMP already exists.
Extracting LMP for Adams
Adams LMP already exists.
Extracting LMP for Adams
Adams LMP already exists.
Extracting LMP for Alder
Extracting LMP for Arne
Extracting LMP for Arne
Arne LMP already exists.
Extracting LMP for Arne
Arne LMP already exists.
Extracting LMP for Arne
Arne LMP already exists.
Extracting LMP for Arthur
Extracting LMP for Arthur
Arthur LMP already exists.
Extracting LMP for Arthur
Arthur LMP already exists.
Extracting LMP for Asser
Extracting LMP for Astor
Extracting LMP for Austen
Extracting LMP for Austen
Austen LMP already exists.
Extracting LMP for Austen
Austen LMP already exists.
Extracting LMP for Austen
Austen LMP already exists.
Extracting LMP for Austen
Austen LMP already exists.
Extracting LMP for Bach
Extractin

In [64]:
# Here, iterate over all the fossil generators to get dispatch and LMP

# Dispatch
for idx, key in enumerate(list(gen_dict["fossil"].keys())):
    if idx == 0:
        make_dispatch_csv(dispatch_path=None, gen_details_path="../Data/thermal_detail.csv", gen_name=key, gen_type="fossil", other_info=None)
    else:
        make_dispatch_csv(dispatch_path="Generator_Dispatch.csv", gen_details_path="../Data/thermal_detail.csv", gen_name=key, gen_type="fossil", other_info=None)

Empty path, make a new df
Extracting LMP for 101_CT_2
Extracting LMP for 101_STEAM_3
Extracting LMP for 101_STEAM_4
Extracting LMP for 102_CT_1
Extracting LMP for 102_CT_2
Extracting LMP for 102_STEAM_3
Extracting LMP for 102_STEAM_4
Extracting LMP for 107_CC_1
Extracting LMP for 113_CT_1
Extracting LMP for 113_CT_2
Extracting LMP for 113_CT_3
Extracting LMP for 113_CT_4
Extracting LMP for 115_STEAM_1
Extracting LMP for 115_STEAM_2
Extracting LMP for 115_STEAM_3
Extracting LMP for 116_STEAM_1
Extracting LMP for 118_CC_1
Extracting LMP for 123_STEAM_2
Extracting LMP for 123_STEAM_3
Extracting LMP for 123_CT_1
Extracting LMP for 123_CT_4
Extracting LMP for 123_CT_5
Extracting LMP for 201_CT_1
Extracting LMP for 201_CT_2
Extracting LMP for 201_STEAM_3
Extracting LMP for 202_CT_1
Extracting LMP for 202_CT_2
Extracting LMP for 202_STEAM_3
Extracting LMP for 202_STEAM_4
Extracting LMP for 207_CT_1
Extracting LMP for 207_CT_2
Extracting LMP for 213_CC_3
Extracting LMP for 213_CT_1
Extracting 

## Check the Summation

In [65]:
# You need to replace the path to the file you saved.
dispatch_path = "Generator_Dispatch.csv"
lmp_path = "Bus_LMP.csv"

df_lmp = pd.read_csv(lmp_path)
df_dispatch = pd.read_csv(dispatch_path)

print(df_lmp.columns)
print(df_dispatch.columns)

Index(['Datetime', 'Abel_LMP', 'Abel_LMP_DA', 'Adams_LMP', 'Adams_LMP_DA',
       'Alder_LMP', 'Alder_LMP_DA', 'Arne_LMP', 'Arne_LMP_DA', 'Arthur_LMP',
       'Arthur_LMP_DA', 'Asser_LMP', 'Asser_LMP_DA', 'Astor_LMP',
       'Astor_LMP_DA', 'Austen_LMP', 'Austen_LMP_DA', 'Bach_LMP',
       'Bach_LMP_DA', 'Bacon_LMP', 'Bacon_LMP_DA', 'Baker_LMP', 'Baker_LMP_DA',
       'Barlow_LMP', 'Barlow_LMP_DA', 'Barton_LMP', 'Barton_LMP_DA',
       'Basov_LMP', 'Basov_LMP_DA', 'Bayle_LMP', 'Bayle_LMP_DA', 'Behring_LMP',
       'Behring_LMP_DA', 'Bloch_LMP', 'Bloch_LMP_DA', 'Cabell_LMP',
       'Cabell_LMP_DA', 'Cabot_LMP', 'Cabot_LMP_DA', 'Carew_LMP',
       'Carew_LMP_DA', 'Cecil_LMP', 'Cecil_LMP_DA', 'Chase_LMP',
       'Chase_LMP_DA', 'Chifa_LMP', 'Chifa_LMP_DA', 'Clark_LMP',
       'Clark_LMP_DA', 'Cobb_LMP', 'Cobb_LMP_DA', 'Cole_LMP', 'Cole_LMP_DA',
       'Comte_LMP', 'Comte_LMP_DA'],
      dtype='object')
Index(['Datetime', '101_CT_1_Dispatch', '101_CT_1_Dispatch DA',
       '101_CT_2_Dispat

In [67]:
# get the summation of the total dispatch of certain generator
gen_name = "101_CT_1"
df_dispatch[gen_name+"_Dispatch"].sum()

5619.0